In [28]:
from langchain_groq import ChatGroq

In [ ]:
from langchain_groq import ChatGroq
import os

llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0,
    groq_api_key=os.getenv('GROQ_API_KEY')
)

llm.invoke("The first person to land on moon was ....").content

'The first person to land on the moon was Neil Armstrong. He stepped out of the lunar module Eagle and onto the moon\'s surface on July 20, 1969, during the Apollo 11 mission. Armstrong famously declared, "That\'s one small step for man, one giant leap for mankind," as he became the first human to set foot on the moon.'

In [30]:
from langchain_community.document_loaders import WebBaseLoader

loader=WebBaseLoader("https://jobright.ai/jobs/info/69a44345b600907a962a389b?utm_source=1121&imp_id=3667457648293191244__instant_push__1772374220962_9917&utm_medium=email")
page_data=loader.load().pop().page_content
print(page_data)

Data Scientist @ Cushman & Wakefield | Jobright.aiSIGN INJOIN NOWData Scientist jobs in United StatesOverviewCompanyApply on Employer SiteAPPLY NOWCushman & Wakefield · 6 hours agoData ScientistCreve Coeur, MOFull-timeOnsiteMid, Senior Level$72K/yr - $85K/yrCushman & Wakefield is a leading global real estate services firm, and they are seeking a Data Scientist to help architect and lead the development of enterprise-scale data platforms and advanced analytics solutions. The role involves partnering with business leadership to apply AI and ML, managing a team, and driving innovation in data engineering.Real EstateConsultingLegalAutomotiveCommercialCommercial Real EstateLeasingProperty ManagementH1B Sponsor LikelyResponsibilitiesPartner with business leadership to identify impactful areas for the application of AI and MLLead Data science and Data driven Analytics engagements across clients covering various sectorsDesign and optimize ML models for predictive analytics and prescriptive ana

In [31]:
from langchain_core.prompts import PromptTemplate

prompt_extract=PromptTemplate.from_template(
    """
    ### SCRAPED TEXT FROM WEBSITE:
    {page_data}
    ### INSTRUCTION
    The scraped text is from the job posting page of a website
    Your job is to extract the job postings and return them in JSON fromat containing following keys: 'role', 'experience','skills' and 'description'.
    Only return the valid JSON.
    ### VALID JSON (NO PREAMBLE): 
    """
)

chain_extract=prompt_extract | llm
res=chain_extract.invoke(input={'page_data':page_data})
print(res.content)

```json
{
  "role": "Data Scientist",
  "experience": "Mid, Senior Level",
  "skills": [
    "Machine Learning",
    "Python",
    "SQL",
    "Databricks",
    "Data Engineering"
  ],
  "description": "Cushman & Wakefield is seeking a Data Scientist to help architect and lead the development of enterprise-scale data platforms and advanced analytics solutions. The role involves partnering with business leadership to apply AI and ML, managing a team, and driving innovation in data engineering."
}
```


In [32]:
from langchain_core.output_parsers import JsonOutputParser

json_parser=JsonOutputParser()
json_res=json_parser.parse(res.content)

In [33]:
json_res

{'role': 'Data Scientist',
 'experience': 'Mid, Senior Level',
 'skills': ['Machine Learning',
  'Python',
  'SQL',
  'Databricks',
  'Data Engineering'],
 'description': 'Cushman & Wakefield is seeking a Data Scientist to help architect and lead the development of enterprise-scale data platforms and advanced analytics solutions. The role involves partnering with business leadership to apply AI and ML, managing a team, and driving innovation in data engineering.'}

In [34]:
type(json_res)

dict

In [35]:
import pandas as pd

df=pd.read_csv("app/resource/portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular, .NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


In [36]:
import chromadb
import uuid


In [44]:
client=chromadb.PersistentClient('vectorstore')
collection=client.get_or_create_collection(name='portfolio')

if not collection.count():
    for _,row in df.iterrows():
        collection.add(documents=[row['Techstack']],
                        metadatas=[{"links": row["Links"]}],
                        ids=[str(uuid.uuid4())]
)

In [46]:
result = collection.query(
    query_texts=["Experience in Python", "Expertise in React Native"],
    n_results=2
)

links = result["metadatas"]
print(links)

[[{'links': 'https://example.com/ml-python-portfolio'}, {'links': 'https://example.com/python-portfolio'}], [{'links': 'https://example.com/react-native-portfolio'}, {'links': 'https://example.com/react-portfolio'}]]


In [47]:
job=json_res
job['skills']

['Machine Learning', 'Python', 'SQL', 'Databricks', 'Data Engineering']

In [48]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}
        
        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools. 
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability, 
        process optimization, cost reduction, and heightened overall efficiency. 
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ 
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ. 
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):
        
        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert Data Science Solutions for Cushman & Wakefield

Dear Hiring Manager,

I came across the job posting for a Data Scientist at Cushman & Wakefield, and I'm excited to introduce AtliQ, an AI & Software Consulting company that can help you architect and lead the development of enterprise-scale data platforms and advanced analytics solutions.

At AtliQ, we have a proven track record of empowering enterprises with tailored solutions, fostering scalability, process optimization, cost reduction, and heightened overall efficiency. Our team of experts has extensive experience in Machine Learning, Python, SQL, Databricks, and Data Engineering, making us an ideal partner to drive innovation in data engineering and apply AI and ML to your business.

Our portfolio showcases our capabilities in developing cutting-edge solutions, including:
- Machine Learning and Python-based projects: https://example.com/ml-python-portfolio
- Python-based solutions: https://example.com/python-portfolio